# Runbook 3.4.b: Firewall rules and access control lists (ACLs)

**Skill:** Mitigate Risk · **Maps to:** 3.4.B, 3.4.C, 3.4.D · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Implement a firewall ACL evaluator. Given an ordered list of rules and a list of packets, decide
the verdict ("allow" or "deny") for each packet using first-match-wins evaluation with an implicit
deny-all at the end of the list.

A **rule** is a dict with these string/number keys:
- `"action"`, `"allow"` or `"deny"`
- `"src"`: source IP string, or `"any"`
- `"dst"`: destination IP string, or `"any"`
- `"port"`: destination port integer, or `0` meaning any port
- `"proto"`: protocol string: `"tcp"`, `"udp"`, or `"any"`

A **packet** is a dict with:
- `"src"`: source IP string
- `"dst"`: destination IP string
- `"port"`: destination port integer
- `"proto"`: protocol string (`"tcp"` or `"udp"`)

**Lesson objective.** Write and evaluate an ordered firewall access control list (ACL), correctly apply first-match-wins evaluation, and identify when a rule's placement creates a security gap by shadowing a more specific rule below it.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
rules = [
    {"action": "allow", "src": "10.0.4.0", "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},
    {"action": "deny",  "src": "any",      "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},
]
packets = [
    {"src": "10.0.4.0", "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},   # matches rule 1 -> allow
    {"src": "10.0.9.9", "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},   # matches rule 2 -> deny
    {"src": "10.0.9.9", "dst": "10.0.1.5", "port": 22,   "proto": "tcp"},   # matches nothing -> deny
]
solve(rules, packets) == ["allow", "deny", "deny"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(rules: list[dict], packets: list[dict]) -> list[str]
```

#### Input

##### `rules`, list of rule dicts, evaluated top-to-bottom (first match wins)
Each rule dict has:
- `"action"` (str): `"allow"` or `"deny"`
- `"src"` (str): source IP address, or `"any"`
- `"dst"` (str): destination IP address, or `"any"`
- `"port"` (int): destination port, or `0` meaning any port
- `"proto"` (str): `"tcp"`, `"udp"`, or `"any"`

##### `packets`, list of packet dicts, evaluated independently, in order
Each packet dict has:
- `"src"` (str): source IP address
- `"dst"` (str): destination IP address
- `"port"` (int): destination port
- `"proto"` (str): `"tcp"` or `"udp"`

#### Output (`result`)
`list[str]` of verdicts, one per packet, each `"allow"` or `"deny"`, in the same order as
`packets`.

#### Rule matching
A rule matches a packet when ALL hold:
1. `rule["src"] == "any"` OR `rule["src"] == packet["src"]`
2. `rule["dst"] == "any"` OR `rule["dst"] == packet["dst"]`
3. `rule["port"] == 0` OR `rule["port"] == packet["port"]`
4. `rule["proto"] == "any"` OR `rule["proto"] == packet["proto"]`

Evaluation returns the **first** matching rule's action. If no rule matches, the implicit
deny-all applies: verdict = `"deny"`.

#### Guarantees / rules
- Pure function; deterministic; no I/O; no network access.
- `rules` list is not mutated by `solve`.
- Standard library only.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 3.4.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented.
"""
from typing import List, Dict


def solve(rules: List[Dict], packets: List[Dict]) -> List[str]:
    """Evaluate each packet against the ACL using first-match-wins logic.

    Args:
        rules:   Ordered list of ACL rule dicts (action, src, dst, port, proto).
        packets: List of packet dicts (src, dst, port, proto).

    Returns:
        List of 'allow' / 'deny' verdicts, one per packet, in order.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_basic_allow_and_deny():
    rules = [
        {"action": "allow", "src": "10.0.4.0", "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},
        {"action": "deny", "src": "any", "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},
    ]
    packets = [
        {"src": "10.0.4.0", "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},
        {"src": "10.0.9.9", "dst": "10.0.1.5", "port": 3306, "proto": "tcp"},
        {"src": "10.0.9.9", "dst": "10.0.1.5", "port": 22, "proto": "tcp"},
    ]
    assert impl.solve(rules, packets) == ["allow", "deny", "deny"]


def test_shadowing_broad_rule_first():
    rules = [
        {"action": "allow", "src": "any", "dst": "any", "port": 22, "proto": "tcp"},
        {"action": "deny", "src": "203.0.113.0", "dst": "10.0.1.10", "port": 22, "proto": "tcp"},
    ]
    packet = {"src": "203.0.113.0", "dst": "10.0.1.10", "port": 22, "proto": "tcp"}
    assert impl.solve(rules, [packet]) == ["allow"]


def test_specific_rule_first_fixes_shadowing():
    rules = [
        {"action": "deny", "src": "203.0.113.0", "dst": "10.0.1.10", "port": 22, "proto": "tcp"},
        {"action": "allow", "src": "any", "dst": "any", "port": 22, "proto": "tcp"},
    ]
    packets = [
        {"src": "203.0.113.0", "dst": "10.0.1.10", "port": 22, "proto": "tcp"},
        {"src": "198.51.100.1", "dst": "10.0.1.10", "port": 22, "proto": "tcp"},
    ]
    assert impl.solve(rules, packets) == ["deny", "allow"]


def test_any_fields_match_everything():
    rules = [
        {"action": "allow", "src": "any", "dst": "any", "port": 0, "proto": "any"},
    ]
    packets = [
        {"src": "1.1.1.1", "dst": "2.2.2.2", "port": 9999, "proto": "udp"},
        {"src": "8.8.8.8", "dst": "9.9.9.9", "port": 1, "proto": "tcp"},
    ]
    assert impl.solve(rules, packets) == ["allow", "allow"]


def test_proto_mismatch_falls_through_to_implicit_deny():
    rules = [
        {"action": "allow", "src": "any", "dst": "any", "port": 53, "proto": "udp"},
    ]
    packets = [
        {"src": "10.0.0.2", "dst": "8.8.8.8", "port": 53, "proto": "tcp"},
    ]
    assert impl.solve(rules, packets) == ["deny"]


def test_empty_rules_denies_everything():
    packets = [{"src": "1.1.1.1", "dst": "2.2.2.2", "port": 80, "proto": "tcp"}]
    assert impl.solve([], packets) == ["deny"]


def test_empty_packets():
    rules = [{"action": "allow", "src": "any", "dst": "any", "port": 0, "proto": "any"}]
    assert impl.solve(rules, []) == []


def test_multiple_packets_order_preserved():
    rules = [
        {"action": "allow", "src": "any", "dst": "any", "port": 443, "proto": "tcp"},
        {"action": "allow", "src": "any", "dst": "any", "port": 80, "proto": "tcp"},
        {"action": "deny", "src": "any", "dst": "any", "port": 0, "proto": "any"},
    ]
    packets = [
        {"src": "a", "dst": "b", "port": 443, "proto": "tcp"},
        {"src": "a", "dst": "b", "port": 21, "proto": "tcp"},
        {"src": "a", "dst": "b", "port": 80, "proto": "tcp"},
    ]
    assert impl.solve(rules, packets) == ["allow", "deny", "allow"]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it